### Ingest the data

### use wines-cluster

In [12]:
try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    pass  # already available in Databricks UI

username = "doman.mat@gmail.com"
schema = "wines/tables"
file_name = "wine_reviews-2026-04-11-23-44.csv"

file_path = f"/Workspace/Users/{username}/{schema}/{file_name}"

df = spark.read.csv(file_path, header=True, inferSchema=True, multiLine=True, escape='"')

In [10]:
import re

def clean_csv_newlines(src_path: str) -> str:
    dst_path = src_path.replace(".csv", "-cleaned.csv")
    with open(src_path, encoding="utf-8") as f:
        content = f.read()
    cleaned = re.sub(
        r'"([^"]*)"',
        lambda m: '"' + re.sub(r'[\r\n]+', ' ', m.group(1)) + '"',
        content,
        flags=re.DOTALL,
    )
    with open(dst_path, "w", encoding="utf-8", newline="") as f:
        f.write(cleaned)
    print(f"Cleaned CSV written to: {dst_path}")
    return dst_path
# no newlines in the reviews in csv now 
clean_path = clean_csv_newlines(file_path)


df = spark.read.csv(clean_path, header=True, inferSchema=True)

FileNotFoundError: [Errno 2] No such file or directory: '/Workspace/Users/doman.mat@gmail.com/wines/tables/wine_reviews-2026-04-11-23-44.csv'

In [13]:
print(f"Row count: {df.count():,}")
df.printSchema()

Row count: 135,211
root
 |-- name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- company: string (nullable = true)
 |-- vintage: string (nullable = true)
 |-- drink_type: string (nullable = true)
 |-- wine_type: string (nullable = true)
 |-- varietal_label: string (nullable = true)
 |-- alcohol: double (nullable = true)
 |-- bottle_size: string (nullable = true)
 |-- case_production: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- appellation: string (nullable = true)
 |-- designation: string (nullable = true)
 |-- retail: double (nullable = true)
 |-- rating: integer (nullable = true)
 |-- reviewer: string (nullable = true)
 |-- review: string (nullable = true)
 |-- date_of_review: date (nullable = true)
 |-- date_received: date (nullable = true)
 |-- pub_date_web: date (nullable = true)
 |-- slug: string (nullable = true)



In [14]:
display(df.limit(10))

,name,brand,company,vintage,drink_type,wine_type,varietal_label,alcohol,bottle_size,case_production,country,state,appellation,designation,retail,rating,reviewer,review,date_of_review,date_received,pub_date_web,slug
0,ZD 2017 Founder's Reserve Pinot Noir (Carneros),ZD,ZD Wines,2017,wine,Red,Pinot Noir,0.1420,750 ml,600,USA,California,Carneros,Founder's Reserve,90.0,91,V.B.,"A fruity, inviting nose of dark cherry and baking spice lead the way to a complex, textured and richly layered midpalate of integrated intensity and dried herb in this reserve-worthy wine. Robust and hearty on the palate it retains a welcome freshness and lasting note of sweet oak.",2019-11-01,2019-07-25,2020-03-01,zd-2017-founders-reserve-pinot-noir-carneros
1,Zaca Mesa 2017 Bien Nacido Vineyard Pinot Noir (Santa Maria Valley),Zaca Mesa,Zaca Mesa Winery,2017,wine,Red,Pinot Noir,0.1410,750 ml,501,USA,California,Santa Maria Valley,Bien Nacido Vineyard,48.0,93,M.K.,"Broad black-cherry and macerated strawberry aromas meet with plenty of clove and cardamom on the soft, lush nose of this bottling. The palate offers a broad range of flavors from Montmorency cherry to white pepper, eucalyptus and bay leaf.",2019-10-01,2019-10-11,2020-09-01,zaca-mesa-2017-bien-nacido-vineyard-pinot-noir-santa-maria-valley
2,YNOT 2017 Pinot Noir (Santa Barbara County),YNOT,Flying Goat Cellars,2017,wine,Red,Pinot Noir,0.1423,750 ml,176,USA,California,Santa Barbara County,NaN,28.0,92,M.K.,"Mulberry, macerated wild cherry, cherry pit and dusty garam masala spices show on the nose of this bottling, Norm Yost's affordable appellation blend. There's a black-plum tang to the sip, where cardamom and curry spices decorate the roasted tomato and black-cherry flavors.",2020-05-02,2020-05-22,2020-10-01,ynot-2017-pinot-noir-santa-barbara-county
3,Writer's Block 2017 Pinot Noir (Lake County),Writer's Block,Steele Wines,2017,wine,Red,Pinot Noir,0.1350,750 ml,1050,USA,California,Lake County,NaN,18.0,88,J.G.,"A good choice in a pleasant, well-mannered wine, this bottle offers just-ripe black-cherry and cinnamon flavors, medium body and a smooth texture.",2020-01-03,2019-12-14,2020-05-01,writers-block-2017-pinot-noir-lake-county
4,Work Vineyard 2017 Zina's Pinot Noir (Russian River Valley),Work Vineyard,Work Vineyard,2017,wine,Red,Pinot Noir,0.1470,750 ml,150,USA,California,Russian River Valley,Zina's,55.0,88,V.B.,"This hearty, thick wine shows the warmth of the vintage, thick in cinnamon cola and waves of full-bodied blueberry lushness.",2020-07-21,2020-07-27,2020-12-01,work-vineyard-2017-zinas-pinot-noir-russian-river-valley
5,Windrun 2017 No. 15 Pinot Noir (Sta. Rita Hills),Windrun,Windrun Wines,2017,wine,Red,Pinot Noir,0.1410,750 ml,350,USA,California,Sta. Rita Hills,No. 15,33.0,92,M.K.,"Beautifully candied dark-cherry aromas meet with vanilla, nutmeg and cardamom on the lush, inviting nose of this bottling. The sip snaps with strong raspberry and rose-petals flavors that ride a very fresh acidity into a lively finish.",2020-01-18,2020-01-24,2020-05-01,windrun-2017-no-15-pinot-noir-sta-rita-hills
6,Windchaser 2017 Pinot Noir (Anderson Valley),Windchaser,Windchaser Wine Co.,2017,wine,Red,Pinot Noir,0.1330,750 ml,300,USA,California,Anderson Valley,NaN,35.0,91,J.G.,"Nicely ripe and well balanced at the same time, this full-bodied wine is scented with cherries and baking spices. Fermented with native yeast, it brings a good grip of tannins and acidity to the palate.",2020-03-25,2020-03-14,2020-08-01,windchaser-2017-pinot-noir-anderson-valley
7,Willowbrook 2017 Kaufman Sunnyslope Vineyard Clone 777 Pinot Noir (Sonoma Mountain),Willowbrook,Willowbrook Cellars,2017,wine,Red,Pinot Noir,0.1430,750 ml,480,USA,California,Sonoma Mountain,Kaufman Sunnyslope Vineyard Clone 777,28.0,91,V.B.,"Flavors of black tea, strawberry and black cherry ride along a smooth, velvety palate. While well-structured, it also retains plenty of lift and delicacy throughout.",2020-04-02,2019-12-12,2020-09-01,willowbrook-2017-kaufman

In [0]:
display(df)

Databricks data profile. Run in Databricks to view.

# Save as Delta Bronze

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("wine_reviews_bronze")

print("Bronze table written.")

## Cleaning and casting 

In [17]:
from pyspark.sql.functions import col, expr, when, sum as spark_sum

cast_columns = [
    ("alcohol",         "double"),
    ("vintage",         "int"),
    ("case_production", "long"),
    ("retail",          "double"),
    ("rating",          "int"),
]

# Cache so the CSV is parsed once and reused for all subsequent actions
df.cache()

# Single agg pass — counts nulls for all columns at once instead of one .count() per column
nulls_before = df.agg(*[
    spark_sum(col(c).isNull().cast("int")).alias(c) for c, _ in cast_columns
]).collect()[0]

# Build all transformations lazily — no Spark action triggered yet
df_cast = df
for column, dtype in cast_columns:
    df_cast = (df_cast
        .withColumn(f"{column}_cast", expr(f"try_cast({column} as {dtype})"))
        # keep original string in _error column only where cast failed
        .withColumn(f"{column}_error", when(
            col(f"{column}_cast").isNull() & col(column).isNotNull(), col(column)
        ))
        .withColumn(column, col(f"{column}_cast"))
        .drop(f"{column}_cast")
    )

# 1 for NV wines, 0 for everything else (genuine vintage or other cast errors)
# vintage stays null for both NV and other bad values — is_nv distinguishes the reason
df_cast = df_cast.withColumn("is_nv", when(col("vintage_error") == "NV", 1).otherwise(0))

# Cache cast result so the transformation chain isn't recomputed per column report
df_cast.cache()

# Second single agg pass — counts nulls after casting for all columns at once
nulls_after = df_cast.agg(*[
    spark_sum(col(c).isNull().cast("int")).alias(c) for c, _ in cast_columns
]).collect()[0]

for column, _ in cast_columns:
    before = nulls_before[column] or 0
    after = nulls_after[column] or 0
    failures = after - before
    print(f"{column:20s}  nulls before={before:>6,}  after={after:>6,}  failures={failures:>6,}")
    if failures > 0:
        df_cast.filter(col(f"{column}_error").isNotNull()).select(f"{column}_error").distinct().show()

alcohol               nulls before=     0  after=     0  failures=     0
vintage               nulls before=     0  after= 4,619  failures= 4,619
+-------------+
|vintage_error|
+-------------+
|           NV|
+-------------+

case_production       nulls before=     0  after=     0  failures=     0
retail                nulls before=     0  after=     0  failures=     0
rating                nulls before=     0  after=     0  failures=     0


In [18]:
display(df)

,name,brand,company,vintage,drink_type,wine_type,varietal_label,alcohol,bottle_size,case_production,country,state,appellation,designation,retail,rating,reviewer,review,date_of_review,date_received,pub_date_web,slug
0,ZD 2017 Founder's Reserve Pinot Noir (Carneros),ZD,ZD Wines,2017,wine,Red,Pinot Noir,0.1420,750 ml,600,USA,California,Carneros,Founder's Reserve,90.0,91,V.B.,"A fruity, inviting nose of dark cherry and baking spice lead the way to a complex, textured and richly layered midpalate of integrated intensity and dried herb in this reserve-worthy wine. Robust and hearty on the palate it retains a welcome freshness and lasting note of sweet oak.",2019-11-01,2019-07-25,2020-03-01,zd-2017-founders-reserve-pinot-noir-carneros
1,Zaca Mesa 2017 Bien Nacido Vineyard Pinot Noir (Santa Maria Valley),Zaca Mesa,Zaca Mesa Winery,2017,wine,Red,Pinot Noir,0.1410,750 ml,501,USA,California,Santa Maria Valley,Bien Nacido Vineyard,48.0,93,M.K.,"Broad black-cherry and macerated strawberry aromas meet with plenty of clove and cardamom on the soft, lush nose of this bottling. The palate offers a broad range of flavors from Montmorency cherry to white pepper, eucalyptus and bay leaf.",2019-10-01,2019-10-11,2020-09-01,zaca-mesa-2017-bien-nacido-vineyard-pinot-noir-santa-maria-valley
2,YNOT 2017 Pinot Noir (Santa Barbara County),YNOT,Flying Goat Cellars,2017,wine,Red,Pinot Noir,0.1423,750 ml,176,USA,California,Santa Barbara County,NaN,28.0,92,M.K.,"Mulberry, macerated wild cherry, cherry pit and dusty garam masala spices show on the nose of this bottling, Norm Yost's affordable appellation blend. There's a black-plum tang to the sip, where cardamom and curry spices decorate the roasted tomato and black-cherry flavors.",2020-05-02,2020-05-22,2020-10-01,ynot-2017-pinot-noir-santa-barbara-county
3,Writer's Block 2017 Pinot Noir (Lake County),Writer's Block,Steele Wines,2017,wine,Red,Pinot Noir,0.1350,750 ml,1050,USA,California,Lake County,NaN,18.0,88,J.G.,"A good choice in a pleasant, well-mannered wine, this bottle offers just-ripe black-cherry and cinnamon flavors, medium body and a smooth texture.",2020-01-03,2019-12-14,2020-05-01,writers-block-2017-pinot-noir-lake-county
4,Work Vineyard 2017 Zina's Pinot Noir (Russian River Valley),Work Vineyard,Work Vineyard,2017,wine,Red,Pinot Noir,0.1470,750 ml,150,USA,California,Russian River Valley,Zina's,55.0,88,V.B.,"This hearty, thick wine shows the warmth of the vintage, thick in cinnamon cola and waves of full-bodied blueberry lushness.",2020-07-21,2020-07-27,2020-12-01,work-vineyard-2017-zinas-pinot-noir-russian-river-valley
5,Windrun 2017 No. 15 Pinot Noir (Sta. Rita Hills),Windrun,Windrun Wines,2017,wine,Red,Pinot Noir,0.1410,750 ml,350,USA,California,Sta. Rita Hills,No. 15,33.0,92,M.K.,"Beautifully candied dark-cherry aromas meet with vanilla, nutmeg and cardamom on the lush, inviting nose of this bottling. The sip snaps with strong raspberry and rose-petals flavors that ride a very fresh acidity into a lively finish.",2020-01-18,2020-01-24,2020-05-01,windrun-2017-no-15-pinot-noir-sta-rita-hills
6,Windchaser 2017 Pinot Noir (Anderson Valley),Windchaser,Windchaser Wine Co.,2017,wine,Red,Pinot Noir,0.1330,750 ml,300,USA,California,Anderson Valley,NaN,35.0,91,J.G.,"Nicely ripe and well balanced at the same time, this full-bodied wine is scented with cherries and baking spices. Fermented with native yeast, it brings a good grip of tannins and acidity to the palate.",2020-03-25,2020-03-14,2020-08-01,windchaser-2017-pinot-noir-anderson-valley
7,Willowbrook 2017 Kaufman Sunnyslope Vineyard Clone 777 Pinot Noir (Sonoma Mountain),Willowbrook,Willowbrook Cellars,2017,wine,Red,Pinot Noir,0.1430,750 ml,480,USA,California,Sonoma Mountain,Kaufman Sunnyslope Vineyard Clone 777,28.0,91,V.B.,"Flavors of black tea, strawberry and black cherry ride along a smooth, velvety palate. While well-structured, it also retains plenty of lift and delicacy throughout.",2020-04-02,2019-12-12,2020-09-01,willowbrook-2017-kaufman